In [ ]:
import os
# 关键：解决OMP冲突问题，防止内核重启
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
from collections import Counter

# ===================== 1. 设备配置 =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# ===================== 2. 加载数据集 =====================
df = pd.read_csv("weibo_senti_100k.csv")
print(f"数据集总量: {len(df)}")

# 文本清洗
def clean_weibo_text(text):
    text = re.sub(r"http\S+", "", str(text))  # 去除URL
    text = re.sub(r"@\w+", "", text)          # 去除@用户名
    text = re.sub(r"\[.*?\]", "", text)       # 去除表情
    text = re.sub(r"[^\w\s]", "", text)       # 去除特殊符号
    return text.strip()

df["clean_text"] = df["review"].apply(clean_weibo_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

# 划分训练集/测试集
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"训练集: {len(train_df)}, 测试集: {len(test_df)}")

# ===================== 3. 文本数字化预处理 =====================
# 构建词汇表
def build_vocab(texts, max_vocab=50000):
    all_words = []
    for text in texts:
        all_words.extend(list(text))
    vocab = Counter(all_words).most_common(max_vocab)
    word2idx = {word: i+2 for i, (word, _) in enumerate(vocab)}  # 0=pad, 1=unk
    word2idx["<PAD>"] = 0
    word2idx["<UNK>"] = 1
    return word2idx

# 生成词汇表
word2idx = build_vocab(df["clean_text"].values)
vocab_size = len(word2idx)
print(f"词汇表大小: {vocab_size}")

# 文本转序列
def text_to_sequence(text, word2idx, max_len=100):
    seq = [word2idx.get(word, 1) for word in list(text)]
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# 超参数
MAX_LEN = 100
BATCH_SIZE = 64
EMBEDDING_DIM = 128

# 自定义数据集
class WeiboDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [text_to_sequence(t, word2idx, MAX_LEN) for t in texts]
        self.labels = labels.values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# 数据加载器
train_dataset = WeiboDataset(train_df["clean_text"], train_df["label"])
test_dataset = WeiboDataset(test_df["clean_text"], test_df["label"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ===================== 4. CNN+BiLSTM 模型核心 =====================
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes=2):
        super().__init__()
        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # CNN层：提取局部文本特征
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=embed_dim, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        
        # BiLSTM层：捕捉上下文依赖
        self.bilstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            bidirectional=True,
            batch_first=True,
            num_layers=1
        )
        
        # 分类层
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        # 维度转换：[batch, seq_len] -> [batch, seq_len, embed_dim]
        x = self.embedding(x)
        # CNN要求输入: [batch, embed_dim, seq_len]
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        # 转换回LSTM输入格式
        x = x.permute(0, 2, 1)
        # BiLSTM
        x, _ = self.bilstm(x)
        # 取最后一个时间步
        x = x[:, -1, :]
        x = self.dropout(x)
        return self.fc(x)

# 初始化模型
model = CNNBiLSTM(vocab_size, EMBEDDING_DIM).to(device)

# ===================== 5. 训练配置 =====================
EPOCHS = 15
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ===================== 6. 训练/评估函数 =====================
def train_epoch():
    model.train()
    total_loss, total_acc = 0, 0
    for seq, label in tqdm(train_loader, desc="训练"):
        seq, label = seq.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += accuracy_score(label.cpu(), torch.argmax(output, dim=1).cpu())
    return total_loss/len(train_loader), total_acc/len(train_loader)

def eval_epoch():
    model.eval()
    total_loss, total_acc = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for seq, label in tqdm(test_loader, desc="评估"):
            seq, label = seq.to(device), label.to(device)
            output = model(seq)
            loss = criterion(output, label)
            
            total_loss += loss.item()
            preds = torch.argmax(output, dim=1).cpu()
            all_preds.extend(preds)
            all_labels.extend(label.cpu())
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=["负面", "正面"])
    return total_loss/len(test_loader), acc, report

# ===================== 7. 开始训练 =====================
for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch+1}/{EPOCHS} ========")
    train_loss, train_acc = train_epoch()
    test_loss, test_acc, report = eval_epoch()
    
    print(f"训练 Loss: {train_loss:.4f} | 准确率: {train_acc:.4f}")
    print(f"测试 Loss: {test_loss:.4f} | 准确率: {test_acc:.4f}")
    print(report)

# 保存模型
torch.save(model.state_dict(), "cnn_bilstm_weibo.pth")
print("\n模型已保存：cnn_bilstm_weibo.pth")

# ===================== 8. 单条文本预测 =====================
def predict(text):
    model.eval()
    text = clean_weibo_text(text)
    seq = text_to_sequence(text, word2idx, MAX_LEN)
    seq = torch.tensor([seq], dtype=torch.long).to(device)
    with torch.no_grad():
        output = model(seq)
    res = torch.argmax(output, dim=1).item()
    return "正面 😊" if res == 1 else "负面 😠"

# 测试预测
print("\n===== 预测测试 =====")
print("文本：这部电影太好看了，强烈推荐！→", predict("这部电影太好看了，强烈推荐！"))
print("文本：真差劲，浪费时间！→", predict("真差劲，浪费时间！"))

In [ ]:
# ===================== 单条文本预测（手动输入测试） =====================
def predict_single_text(text):
    model.eval()
    # 文本清洗
    text = clean_weibo_text(text)
    # 转序列
    seq = text_to_sequence(text, word2idx, MAX_LEN)
    seq = torch.tensor([seq], dtype=torch.long).to(device)
    
    with torch.no_grad():
        output = model(seq)
    
    pred_class = torch.argmax(output, dim=1).item()
    sentiment = "✅ 正面情感" if pred_class == 1 else "❌ 负面情感"
    confidence = torch.softmax(output, dim=1).max().item()
    
    print(f"\n📝 输入文本：{text}")
    print(f"🎯 预测结果：{sentiment}")
    print(f"🔍 置信度：{confidence:.4f}\n")

# ==============================================
# 在这里输入你想测试的微博内容！！！
# ==============================================
test_content = "这个电影真差劲，差评！"  
predict_single_text(test_content)      